In [1]:
import json
import os
from datetime import datetime, timedelta
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import matplotlib.pyplot as plt
from prophet import Prophet

# --- Configuration ---
plt.style.use('seaborn-v0_8')
px.defaults.template = 'plotly_white'

CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)

# EIA Constants
EIA_API_KEY = os.getenv('EIA_API_KEY')
EIA_BASE_URL = 'https://api.eia.gov/v2/electricity/rto/'
HISTORICAL_DAYS = 90
PROPHET_LOOKBACK_DAYS = 60
FORECAST_HOURS = 24 * 7

# Codes for "Carbon-Free" energy sources in EIA data
GREEN_CODES = {'SUN', 'WND', 'WAT', 'GEO', 'NUC'}

# --- Helper: Cache Path ---
def _cache_path(region: str, type: str) -> Path:
    return CACHE_DIR / f'eia_{region.lower()}_{type}.csv'

## EIA Hourly Demand Fetching


In [2]:
# --- 1. Fetch Real Hourly Demand ---
@lru_cache(maxsize=None)
def fetch_eia_hourly(region: str) -> pd.DataFrame:
    """Fetch hourly demand (MW) for the region."""
    cache_file = _cache_path(region, 'demand')
    
    # Check cache first (optional, for development speed)
    if cache_file.exists():
        return pd.read_csv(cache_file, parse_dates=['datetime'])

    url = EIA_BASE_URL + 'region-data/data/'
    end = datetime.utcnow()
    start = end - timedelta(days=HISTORICAL_DAYS)
    
    params = {
        'api_key': EIA_API_KEY,
        'data[0]': 'value',
        'facets[respondent][]': region,
        'frequency': 'hourly',
        'start': start.strftime('%Y-%m-%dT%H'),
        'end': end.strftime('%Y-%m-%dT%H'),
        'sort[0][column]': 'period',
        'sort[0][direction]': 'desc',
        'length': 5000,
    }
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json().get('response', {}).get('data', [])
        
        records = []
        for item in data:
            if item.get('value'):
                records.append({
                    'datetime': pd.to_datetime(item['period']),
                    'demand_MW': float(item['value']),
                    'region': region
                })
        
        df = pd.DataFrame(records)
        if not df.empty:
            df = df.drop_duplicates('datetime').sort_values('datetime')
            df.to_csv(cache_file, index=False)
            return df
        return pd.DataFrame()
        
    except Exception as e:
        print(f"⚠️ Demand fetch failed for {region}: {e}")
        return pd.DataFrame()

# --- 2. Fetch Real Fuel Mix (NEW) ---
@lru_cache(maxsize=None)
def fetch_eia_fuelmix(region: str) -> float:
    """
    Fetch the latest fuel mix and return the Carbon-Free Energy % (0-100).
    Returns NaN if data is unavailable.
    """
    url = EIA_BASE_URL + 'fuel-type-data/data/'
    
    # We only need the most recent ~24 hours to get a current snapshot
    end = datetime.utcnow()
    start = end - timedelta(hours=24)
    
    params = {
        'api_key': EIA_API_KEY,
        'data[0]': 'value',
        'facets[respondent][]': region,
        'frequency': 'hourly',
        'start': start.strftime('%Y-%m-%dT%H'),
        'end': end.strftime('%Y-%m-%dT%H'),
        'sort[0][column]': 'period',
        'sort[0][direction]': 'desc',
        'length': 500 # Enough to cover all fuel types for last few hours
    }
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json().get('response', {}).get('data', [])
        
        if not data:
            return float('nan')
            
        df = pd.DataFrame(data)
        df['value'] = df['value'].astype(float)
        
        # Take the most recent complete hour
        latest_time = df['period'].max()
        current_mix = df[df['period'] == latest_time]
        
        total_gen = current_mix['value'].sum()
        if total_gen == 0:
            return 0.0
            
        # Sum Carbon-Free Sources (Solar, Wind, Hydro, Geo, Nuclear)
        clean_gen = current_mix[current_mix['fueltype'].isin(GREEN_CODES)]['value'].sum()
        
        # Return Percentage (0-100)
        return (clean_gen / total_gen) * 100.0

    except Exception as e:
        print(f"⚠️ Fuel Mix fetch failed for {region}: {e}")
        return float('nan')

## Forecasting and Grid Metrics

In [3]:
# --- 3. Forecasting & Analysis Helpers ---
def _prepare_series(df: pd.DataFrame) -> pd.DataFrame:
    df = df.set_index('datetime').sort_index()
    # Fill missing hourly gaps
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='h')
    df = df.reindex(full_idx)
    df['demand_MW'] = df['demand_MW'].interpolate(method='time')
    return df

def get_forecast_metrics(df: pd.DataFrame):
    """Returns (peak_forecast, volatility, current_load)"""
    if df.empty:
        return np.nan, np.nan, np.nan
        
    s = _prepare_series(df)
    
    # 1. Volatility (Std Dev of last 24h)
    volatility = s['demand_MW'].rolling(24).std().iloc[-1]
    
    # 2. Current Load (Last available)
    current_load = s['demand_MW'].iloc[-1]

    # 3. Forecast Peak (Prophet)
    # Only run prophet if we have enough history
    if len(s) > 24*14:
        try:
            # Use last 60 days for speed
            recent = s.tail(24 * PROPHET_LOOKBACK_DAYS).reset_index()
            recent.columns = ['ds', 'y']
            
            m = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False)
            m.add_country_holidays(country_name='US')
            m.fit(recent)
            
            future = m.make_future_dataframe(periods=FORECAST_HOURS, freq='h')
            forecast = m.predict(future)
            peak_forecast = forecast['yhat'].max()
        except:
            peak_forecast = current_load # Fallback
    else:
        peak_forecast = current_load

    return peak_forecast, volatility, current_load

# --- 4. Fetch Temperature ---
@lru_cache(maxsize=None)
def fetch_temperature(lat: float, lon: float) -> float:
    try:
        url = 'https://api.open-meteo.com/v1/forecast'
        params = {'latitude': lat, 'longitude': lon, 'daily': 'temperature_2m_mean', 'past_days': 60}
        r = requests.get(url, params=params, timeout=10)
        temps = r.json().get('daily', {}).get('temperature_2m_mean', [])
        return np.mean(temps) if temps else np.nan
    except:
        return np.nan

## Compute Datacenter Scores

In [4]:

# --- Main Execution Loop ---

region_coords = {
    'CAL': (36.5, -119.5), 'CAR': (35.5, -80.0), 'CENT': (38.5, -94.5),
    'FLA': (28.0, -82.0), 'MIDA': (39.0, -77.0), 'MIDW': (42.0, -89.0),
    'NE': (42.5, -72.5), 'NY': (42.9, -75.3), 'NW': (45.5, -120.5),
    'SE': (33.0, -84.0), 'SW': (36.0, -111.5), 'TEN': (36.0, -86.0),
    'TEX': (31.0, -99.0),
}

records = []
print("🚀 Starting GridCast Real-Data Pipeline...")

for region, (lat, lon) in region_coords.items():
    print(f"  Processing {region}...")
    
    # A. Fetch Data
    df_demand = fetch_eia_hourly(region)
    renewable_score = fetch_eia_fuelmix(region) # Returns 0-100 or NaN
    avg_temp = fetch_temperature(lat, lon)
    
    # B. Compute Demand Metrics
    peak, volatility, current_load = get_forecast_metrics(df_demand)
    
    records.append({
        'region': region,
        'load_mw': current_load,
        'peak_forecast_mw': peak,
        'volatility': volatility,
        'renewable_score': renewable_score, # Real Data!
        'avg_temp': avg_temp,
        'lat': lat, 
        'lon': lon
    })

# --- DataFrame Construction & Scoring ---
dc_df = pd.DataFrame(records)

# 1. Handle Missing Data (Fill NaNs with column means to preserve scoring)
dc_df = dc_df.fillna(dc_df.mean(numeric_only=True))

# 2. Normalize Metrics (0 to 1)
# Note: For Load, Peak, Volatility, and Temp -> LOWER is BETTER
#       For Renewable Score -> HIGHER is BETTER
cols_to_norm = ['load_mw', 'peak_forecast_mw', 'volatility', 'renewable_score', 'avg_temp']
for col in cols_to_norm:
    min_v = dc_df[col].min()
    max_v = dc_df[col].max()
    if max_v != min_v:
        dc_df[f'{col}_n'] = (dc_df[col] - min_v) / (max_v - min_v)
    else:
        dc_df[f'{col}_n'] = 0.5

# 3. Calculate Component Scores (0 to 1)

# Profitability Score (40% Weight)
# Components: Low Load (40%), Low Peak Forecast (30%), Low Volatility (30%)
# "1 - x" because lower is better for these metrics
dc_df['profitability'] = (
    0.40 * (1 - dc_df['load_mw_n']) +
    0.30 * (1 - dc_df['peak_forecast_mw_n']) +
    0.30 * (1 - dc_df['volatility_n'])
)

# Sustainability Score (60% Weight)
# Components: High Renewable Score (70%), Low Temperature (30%)
dc_df['sustainability'] = (
    0.70 * dc_df['renewable_score_n'] + 
    0.30 * (1 - dc_df['avg_temp_n'])
)

# 4. Final GridCast Score
dc_df['dc_score'] = 0.40 * dc_df['profitability'] + 0.60 * dc_df['sustainability']

# Sort and Clean
dc_df_final = dc_df.sort_values('dc_score', ascending=False).reset_index(drop=True)
dc_df_final = dc_df_final[['region', 'dc_score', 'profitability', 'sustainability', 
                           'renewable_score', 'load_mw', 'avg_temp', 'lat', 'lon']]

# Save
dc_df_final.to_csv('datacenter_scores_real.csv', index=False)

dc_df_final

🚀 Starting GridCast Real-Data Pipeline...
  Processing CAL...


/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_36150/594704589.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


  Processing CAR...


/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_36150/594704589.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


  Processing CENT...


...

  Processing TEN...


/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_36150/594704589.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


  Processing TEX...


/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_36150/594704589.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


,region,dc_score,profitability,sustainability,renewable_score,load_mw,avg_temp,lat,lon
0,CAR,0.836575,0.836114,0.836882,62.274280,21115.0,14.847761,35.5,-80.0
1,NY,0.819783,0.908414,0.760696,46.294596,17600.0,6.700000,42.9,-75.3
2,TEN,0.743210,0.909664,0.632240,48.673019,16976.0,14.895522,36.0,-86.0
3,NW,0.708350,0.622248,0.765750,49.890434,39325.0,9.137313,45.5,-120.5
4,CENT,0.689809,0.738793,0.657152,49.616232,34120.0,14.356716,38.5,-94.5
5,NE,0.685175,0.951633,0.507536,31.877865,14750.0,8.561194,42.5,-72.5
6,SW,0.601283,1.000000,0.335472,25.881175,11780.0,12.670149,36.0,-111.5
7,SE,0.539537,0.768120,0.387149,34.138589,25223.0,16.265672,33.0,-84.0
8,CAL,0.471823,0.767519,0.274692,27.187905,25379.0,16.683582,36.5,-119.5
9,MIDA,0.340134,0.059213,0.527414,38.825942,89539.0,12.765672,39.0,-77.0


In [ ]:
import json
import os
from datetime import datetime, timedelta
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import matplotlib.pyplot as plt
from prophet import Prophet

# --- Configuration ---
plt.style.use('seaborn-v0_8')
px.defaults.template = 'plotly_white'

CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)

EIA_API_KEY = os.getenv('EIA_API_KEY')
EIA_BASE_URL = 'https://api.eia.gov/v2/electricity/'

# Codes for "Carbon-Free" energy sources
GREEN_CODES = {'SUN', 'WND', 'WAT', 'GEO', 'NUC'}

In [ ]:
# --- 1. Fetch Real Hourly Demand ---
@lru_cache(maxsize=None)
def fetch_eia_hourly(region: str) -> pd.DataFrame:
    """Fetch hourly demand (MW)."""
    url = EIA_BASE_URL + 'rto/region-data/data/'
    end = datetime.utcnow()
    start = end - timedelta(days=90)
    
    params = {
        'api_key': EIA_API_KEY,
        'data[0]': 'value',
        'facets[respondent][]': region,
        'frequency': 'hourly',
        'start': start.strftime('%Y-%m-%dT%H'),
        'end': end.strftime('%Y-%m-%dT%H'),
        'sort[0][column]': 'period',
        'sort[0][direction]': 'desc',
        'length': 5000,
    }
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json().get('response', {}).get('data', [])
        df = pd.DataFrame(data)
        if not df.empty:
            df['datetime'] = pd.to_datetime(df['period'])
            df['demand_MW'] = df['value'].astype(float)
            return df.sort_values('datetime')
        return pd.DataFrame()
    except Exception as e:
        print(f"⚠️ Demand fetch failed for {region}: {e}")
        return pd.DataFrame()

# --- 2. Fetch Real Fuel Mix ---
@lru_cache(maxsize=None)
def fetch_eia_fuelmix(region: str) -> float:
    """Fetch Carbon-Free Energy % (0-100)."""
    url = EIA_BASE_URL + 'rto/fuel-type-data/data/'
    end = datetime.utcnow()
    start = end - timedelta(hours=24)
    
    params = {
        'api_key': EIA_API_KEY,
        'data[0]': 'value',
        'facets[respondent][]': region,
        'frequency': 'hourly',
        'start': start.strftime('%Y-%m-%dT%H'),
        'end': end.strftime('%Y-%m-%dT%H'),
        'length': 500
    }
    
    try:
        r = requests.get(url, params=params, timeout=30)
        data = r.json().get('response', {}).get('data', [])
        if not data: return float('nan')
            
        df = pd.DataFrame(data)
        df['value'] = df['value'].astype(float)
        
        # Get most recent hour
        latest_time = df['period'].max()
        current_mix = df[df['period'] == latest_time]
        
        total = current_mix['value'].sum()
        if total == 0: return 0.0
            
        clean = current_mix[current_mix['fueltype'].isin(GREEN_CODES)]['value'].sum()
        return (clean / total) * 100.0
    except:
        return float('nan')

# --- 3. Fetch Real Price (NEW) ---
@lru_cache(maxsize=None)
def fetch_eia_price(region: str) -> float:
    """Fetch latest Wholesale Price ($/MWh)."""
    url = EIA_BASE_URL + 'wholesale-markets-data/data/'
    end = datetime.utcnow()
    start = end - timedelta(hours=24)
    
    # Note: Mapping Regions to Hubs is complex. 
    # For this prototype, we will use a heuristic: 
    # If specific price data is missing, we fallback to a calculated proxy later.
    # This generic call attempts to get any LMP data for the region.
    params = {
        'api_key': EIA_API_KEY,
        'data[0]': 'value',
        'facets[respondent][]': region,
        'frequency': 'hourly',
        'sort[0][column]': 'period',
        'sort[0][direction]': 'desc',
        'length': 10
    }
    try:
        r = requests.get(url, params=params, timeout=5)
        data = r.json().get('response', {}).get('data', [])
        if data:
            # return average of the last few reported prices
            vals = [float(x['value']) for x in data if x['value']]
            return np.mean(vals) if vals else float('nan')
    except:
        pass
    return float('nan')

# --- 4. Fetch Weather ---
@lru_cache(maxsize=None)
def fetch_temperature(lat, lon):
    try:
        url = 'https://api.open-meteo.com/v1/forecast'
        params = {'latitude': lat, 'longitude': lon, 'daily': 'temperature_2m_mean', 'past_days': 60}
        r = requests.get(url, params=params, timeout=10)
        temps = r.json().get('daily', {}).get('temperature_2m_mean', [])
        return np.mean(temps) if temps else np.nan
    except:
        return np.nan

In [ ]:

# --- MAIN EXECUTION ---
region_coords = {
    'CAL': (36.5, -119.5), 'CAR': (35.5, -80.0), 'CENT': (38.5, -94.5),
    'FLA': (28.0, -82.0), 'MIDA': (39.0, -77.0), 'MIDW': (42.0, -89.0),
    'NE': (42.5, -72.5), 'NY': (42.9, -75.3), 'NW': (45.5, -120.5),
    'SE': (33.0, -84.0), 'SW': (36.0, -111.5), 'TEN': (36.0, -86.0),
    'TEX': (31.0, -99.0),
}

records = []
print("🚀 Fetching Real Data...")

for region, (lat, lon) in region_coords.items():
    # A. Fetch Raw Data
    df_demand = fetch_eia_hourly(region)
    raw_renew = fetch_eia_fuelmix(region)
    raw_price = fetch_eia_price(region)
    raw_temp = fetch_temperature(lat, lon)
    
    # B. Compute Derived Metrics
    if not df_demand.empty:
        raw_load = df_demand['demand_MW'].iloc[-1]
        # Volatility (Standard Deviation of last 24h)
        raw_volatility = df_demand['demand_MW'].tail(24).std()
        # Peak Forecast (Simple max of last 30 days as a proxy for capacity)
        raw_peak = df_demand['demand_MW'].max()
    else:
        raw_load, raw_volatility, raw_peak = np.nan, np.nan, np.nan

    # Fallback logic if Price API returns nothing (common for some regions)
    # We proxy price using Load Stress (High Load = High Price)
    if np.isnan(raw_price) and not np.isnan(raw_load):
        raw_price = (raw_load / 1000) * 2.5 # Rough heuristic $2.50 per GW

    records.append({
        'region': region,
        'lat': lat, 'lon': lon,
        'raw_price': raw_price,
        'raw_load': raw_load,
        'raw_volatility': raw_volatility,
        'raw_peak': raw_peak,
        'raw_renew': raw_renew,
        'raw_temp': raw_temp
    })

# --- DATAFRAME CONSTRUCTION ---
dc_df = pd.DataFrame(records)

# 1. Fill Missing Data (Mean Imputation)
dc_df = dc_df.fillna(dc_df.mean(numeric_only=True))

# 2. Normalize Columns (Store as 'n_column')
# We normalize so 0 is "Bad" and 1 is "Good"
# Price: Lower is better -> 1 - norm
# Load: Lower is better -> 1 - norm
# Volatility: Lower is better -> 1 - norm
# Temp: Lower is better -> 1 - norm
# Renewables: Higher is better -> norm

def normalize(series, invert=False):
    min_v, max_v = series.min(), series.max()
    if max_v == min_v: return 0.5
    norm = (series - min_v) / (max_v - min_v)
    return (1 - norm) if invert else norm

dc_df['n_price'] = normalize(dc_df['raw_price'], invert=True)
dc_df['n_load'] = normalize(dc_df['raw_load'], invert=True)
dc_df['n_volatility'] = normalize(dc_df['raw_volatility'], invert=True)
dc_df['n_temp'] = normalize(dc_df['raw_temp'], invert=True)
dc_df['n_renew'] = normalize(dc_df['raw_renew'], invert=False) # Higher is good

# 3. Calculate Scores
# Profitability (40%): Price, Load, Volatility
dc_df['profitability'] = (
    0.40 * dc_df['n_price'] +
    0.30 * dc_df['n_load'] +
    0.30 * dc_df['n_volatility']
)

# Sustainability (60%): Renewables, Temp
dc_df['sustainability'] = (
    0.70 * dc_df['n_renew'] + 
    0.30 * dc_df['n_temp']
)

# Final Score
dc_df['dc_score'] = 0.40 * dc_df['profitability'] + 0.60 * dc_df['sustainability']

# Sort
dc_df_final = dc_df.sort_values('dc_score', ascending=False).reset_index(drop=True)

print("\n🏆 Top Regions (With Full Data):")
print(dc_df_final[['region', 'dc_score', 'raw_price', 'raw_renew']].head())

# Save EVERYTHING
dc_df_final.to_csv('datacenter_scores_real.csv', index=False)